# Player を継承して自作 AI を作る

Player.choose_command() をオーバーライドする。

In [1]:
%pip install -q jpoke

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\tmtmh\Documents\pokemon\jpoke\.venv\Scripts\python.exe -m pip install --upgrade pip


In [ ]:
from jpoke import Battle, Player
from jpoke.enums import Command

In [3]:
class CustomePlayer(Player):
    def choose_selection(self, battle: Battle) -> list[int]:
        """
        Player.choose_selection() は、チームのポケモンのインデックスを選出順に並べたリストを返す。
        ここでは素早さの高い順に選出する。
        """
        indexes = list(range(len(self.team)))
        speeds = [self.team[i].stats["spe"] for i in indexes]
        order = sorted(indexes, key=lambda i: speeds[i], reverse=True)
        return order[:battle.n_selected]
        
    def choose_command(self, battle: Battle) -> Command:
        """
        Player.choose_command() は、行動を表す Command オブジェクトを返す。
        ここでは、ダメージが最大になる行動を選択する。
        """
        commands = battle.available_commands(self)
        return max(commands, key=lambda command: self._damage(battle, command))
    
    def _damage(self, battle: Battle, command: Command) -> int:
        if not command.is_move:
            return -1

        move = battle.command_to_move(self, command) 
        opponent = battle.opponent(self)   
        damages = battle.calc_damages(
            attacker=battle.get_active(self),
            defender=battle.get_active(opponent),
            move=move,
            critical=move.guaranteed_crit # 確定急所を考慮する
        )
        return damages[0] # 0: 最低ダメージ

In [4]:
# TODO: 2vs2の対戦にする
player1 = CustomePlayer("StrongestMovePlayer")
player1.add_pokemon("ピカチュウ", moves=["でんこうせっか", "かみなり", "なきごえ"])

player2 = Player("Player 2")
player2.add_pokemon("フシギダネ", moves=["たいあたり"])

In [5]:
battle = Battle(player1, player2, seed=1)
battle.play_out(max_turns=100)
winner = battle.winner
print(f"勝者: {winner.username if winner else '引き分け（ターン上限）'}")
battle.print_logs("all")
print("-" * 50)  # print_logs() の出力とその後のprint()を視覚的に区切る

勝者: StrongestMovePlayer
Turn 0 : StrongestMovePlayer :  : バトル開始
Turn 0 : StrongestMovePlayer : ピカチュウ : ピカチュウ 入場
Turn 0 : Player 2 : フシギダネ : フシギダネ 入場
Turn 1 : StrongestMovePlayer : ピカチュウ : かみなり PP -1
Turn 1 : Player 2 : フシギダネ : HP -29 (91/120)
Turn 1 : Player 2 : フシギダネ : まひが付与された
Turn 1 : Player 2 : フシギダネ : たいあたり PP -1
Turn 1 : StrongestMovePlayer : ピカチュウ : HP -21 (89/110)
Turn 2 : StrongestMovePlayer : ピカチュウ : かみなり PP -1
Turn 2 : StrongestMovePlayer : ピカチュウ : 急所に当たった！
Turn 2 : Player 2 : フシギダネ : HP -44 (47/120)
Turn 2 : Player 2 : フシギダネ : たいあたり PP -1
Turn 2 : StrongestMovePlayer : ピカチュウ : HP -21 (68/110)
Turn 3 : StrongestMovePlayer : ピカチュウ : かみなり PP -1
Turn 3 : Player 2 : フシギダネ : HP -30 (17/120)
Turn 3 : Player 2 : フシギダネ : 動けない [まひ]
Turn 4 : StrongestMovePlayer : ピカチュウ : かみなり PP -1
Turn 4 : Player 2 : フシギダネ : HP -17 (0/120)
Turn 4 : StrongestMovePlayer :  : 勝利
Turn 4 : Player 2 :  : 敗北
--------------------------------------------------


試してみよう: move_power() に「相手のタイプ相性が悪い技は除外する」といった
条件を加えると、より賢い方策に発展させられる